In [2]:
# OPTIONAL: use a separate notebook and runtime, never the core project runtime.
# Internet and model/library downloads required; no API key or paid endpoint.
import importlib.util
import subprocess
import sys
# Preserve Colab's preinstalled matched torch/vision/audio stack.
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cpu'])
# Install compatible versions of transformers and sentencepiece
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'sentencepiece'])
import torch
print('Torch runtime:', torch.__version__)
print('If Colab requests a restart, restart this optional runtime and run all cells again.')

Torch runtime: 2.14.1+cpu
If Colab requests a restart, restart this optional runtime and run all cells again.


In [3]:
# OPTIONAL REAL-MODEL EXTENSION: run in a separate notebook/runtime.
# Never paste into or replace cells in the required mock notebook.
import json
import time
from pathlib import Path

LIVE_MODEL_ID = 'google/flan-t5-small'
LIVE_MODEL_REVISION = '0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab'
LIVE_OUTPUT_DIR = Path('murshid_live_outputs')




In [4]:
def run_live_demo():
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    torch.set_num_threads(2)
    tokenizer = AutoTokenizer.from_pretrained(
        LIVE_MODEL_ID, revision=LIVE_MODEL_REVISION, trust_remote_code=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        LIVE_MODEL_ID, revision=LIVE_MODEL_REVISION,
        trust_remote_code=False, use_safetensors=True).to('cpu').eval()
    # Public synthetic context only; no private application records or identities.
    context = 'You can check your application status using its reference number.'
    question = 'How can I check my application status?'
    prompt = f'Answer the question using the context.\nContext: {context}\nQuestion: {question}\nAnswer:'
    encoded = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**encoded, max_new_tokens=48, do_sample=False, num_beams=1)
    answer = tokenizer.decode(generated[0], skip_special_tokens=True).strip()
    # Smoke check only: this does NOT establish factual correctness or safety.
    assert isinstance(answer, str) and answer, 'No answer generated; inspect the runtime.'
    report = {
        'mode': 'real_local_model', 'model': LIVE_MODEL_ID,
        'model_revision': LIVE_MODEL_REVISION, 'device': 'cpu',
        'context': context, 'question': question, 'answer': answer,
        'inference_seconds': round(time.perf_counter() - started, 3),
        'input_tokens': int(encoded['input_ids'].shape[-1]),
        'output_tokens_including_special': int(generated.shape[-1]),
        'external_inference_api_calls': 0,
        'review_required': 'Check whether the answer tells the user to use the reference number. Reject unsupported fees, dates or status claims.'
    }
    LIVE_OUTPUT_DIR.mkdir(exist_ok=True)
    (LIVE_OUTPUT_DIR / 'live_demo_result.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))
    return report




In [5]:
live_result = run_live_demo()
print('REVIEW: Does the answer say to use the reference number?')
print('Reject invented fees, dates or application statuses. Nonempty text is not a quality pass.')


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

{
  "mode": "real_local_model",
  "model": "google/flan-t5-small",
  "model_revision": "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab",
  "device": "cpu",
  "context": "You can check your application status using its reference number.",
  "question": "How can I check my application status?",
  "answer": "Use its reference number",
  "inference_seconds": 0.269,
  "input_tokens": 34,
  "output_tokens_including_special": 6,
  "external_inference_api_calls": 0,
  "review_required": "Check whether the answer tells the user to use the reference number. Reject unsupported fees, dates or status claims."
}
REVIEW: Does the answer say to use the reference number?
Reject invented fees, dates or application statuses. Nonempty text is not a quality pass.


The application combines the supplied context and question into an input, converts the text into tokens, and passes them to the FLAN-T5 model, which generates output tokens that are decoded into an answer.

Unlike FakeClient, which returns a predefined response from a fixed mapping, the real model generates its response using learned model weights.